# Summarize large text

<p style="font-size: 20px; font-weight: bold; font-style: italic;">...via LLM-graph</p>

Anton Antonov   
August, September 2025

----

## Introduction

This notebook illustrates how to specify a Large Language Model (LLM) graph for deriving comprehensive summaries of large texts. 
The LLM graph is based on different LLM- and non-LLM functions .
The Raku package ["LLM::Graph"](https://raku.land/zef:antononcube/LLM::Graph) is used, [AAp1].

Using the LLM graph is an alternative to the Literate programming based solutions shown in [AA1, AAn1].

----

## Setup

In [1]:
use LLM::Graph;
use Data::Importers;

use LLM::Tooling;

In [2]:
sink my $conf5-mini = llm-configuration('ChatGPT', model => 'gpt-5-mini');
sink my $conf5 = llm-configuration('ChatGPT', model => 'gpt-5');
sink my $conf41-mini = llm-configuration('ChatGPT', model => 'gpt-4.1-mini', temperature => 0.55, max-tokens => 4096);
sink my $conf41 = llm-configuration('ChatGPT', model => 'gpt-4.1', temperature => 0.45, max-tokens => 8192);
sink my $conf4o-mini = llm-configuration('ChatGPT', model => 'gpt-4o-mini', temperature => 0.45, max-tokens => 8192);
sink my $conf4o = llm-configuration('ChatGPT', model => 'gpt-4o', temperature => 0.45, max-tokens => 8192);

----

## LLM graph

Specify the LLM graph nodes:

In [3]:
sink my %rules =
TypeOfInput => sub ($_) {
        "Determine the input type of\n\n$_.\n\nThe result should be one of: 'Text', 'URL', 'FilePath', or 'Other'."  ~ 
        llm-prompt('NothingElse')('single string')
    },

IngestText =>  { eval-function => sub ($TypeOfInput, $_) { $TypeOfInput ~~ / URL | FilePath/ ?? data-import($_) !! $_} },

Title => { 
    eval-function => sub ($IngestText, $with-title = Whatever) { $with-title ~~ Str:D ?? $with-title !! llm-synthesize([llm-prompt("TitleSuggest")($IngestText, 'article'), "Short title with less that 6 words"]) },
},

Summary => sub ($IngestText) { llm-prompt("Summarize")() ~ "\n\n$IngestText" },

TopicsTable => sub ($IngestText) { llm-prompt("ThemeTableJSON")($IngestText, 'article', 20) },

ThinkingHats => sub ($IngestText) { llm-prompt("ThinkingHatsFeedback")($IngestText, <yellow grey>, format => 'HTML') },

MindMap => sub ($IngestText) { llm-prompt('MermaidDiagram')($IngestText) },

Report => { eval-function => 
    sub ($Title, $Summary, $TopicsTable, $MindMap, $ThinkingHats) { 
        [
            "# $Title",
            '### *LLM summary report*',
            '## Summary',
            $Summary,
            '## Topics',
            to-html(
                from-json($TopicsTable.subst(/ ^ '```json' | '```' $/):g),
                field-names => <theme content>,
                align => 'left'),
            "## Mind map",
            $MindMap,
            '## Thinking hats',
            $ThinkingHats.subst(/ ^ '```html' | '```' $/):g
        ].join("\n\n")
    } 
},

ExportAndOpen => {
    eval-function => sub ($Report) {
       spurt('./Report.md', $Report);
       shell "open ./Report.md" 
    },
    test-function => -> $export-and-open = True { $export-and-open ~~ Bool:D && $export-and-open || $export-and-open.Str.lc ∈ <true yes open> }
}
;

Make the graph:

In [4]:
my $gCombinedSummary = llm-graph(%rules, llm-evaluator => $conf41-mini, :async, :progress)

LLM::Graph(size => 9, nodes => ExportAndOpen, IngestText, MindMap, Report, Summary, ThinkingHats, Title, TopicsTable, TypeOfInput)

----

## Full computation

URL and text statistics:

In [5]:
my $url = 'https://raw.githubusercontent.com/antononcube/RakuForPrediction-blog/refs/heads/main/Data/Graph-neat-examples-in-Raku-Set-2-YouTube.txt';
my $txtFocus = data-import($url);

text-stats($txtFocus)

(chars => 5957 words => 1132 lines => 157)

Computation:

In [6]:
$gCombinedSummary.eval({'$_' => $url, with-title => '«Graph» neat examples, set 3'})

Awaiting for the results of TypeOfInput
Awaiting for the results of ThinkingHats MindMap TopicsTable Summary


LLM::Graph(size => 9, nodes => ExportAndOpen, IngestText, MindMap, Report, Summary, ThinkingHats, Title, TopicsTable, TypeOfInput)

**Remark:** Instead of deriving the title using an LLM, the title is specified as an argument.

Show the corresponding graph:

In [7]:
#% html
$gCombinedSummary.dot(node-width => 1.2, theme => 'default'):svg

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
 
 TopicsTable_Cluster 
 
 
 
 MindMap_Cluster 
 
 
 
 ThinkingHats_Cluster 
 
 
 
 Summary_Cluster 
 
 
 
 TypeOfInput_Cluster 
 
 
<!-- $_ -->
 
 $_ 
 
 $_ 
 
<!-- IngestText -->
 
 IngestText 
 
 IngestText 
 
<!-- $_->IngestText -->
 
 $_->IngestText 
 
 
 
<!-- TypeOfInput -->
 
 TypeOfInput 
 
 TypeOfInput 
 
<!-- $_->TypeOfInput -->
 
 $_->TypeOfInput 
 
 
 
<!-- ExportAndOpen -->
 
 ExportAndOpen 
 
 ExportAndOpen 
 
<!-- MindMap -->
 
 MindMap 
 
 MindMap 
 
<!-- IngestText->MindMap -->
 
 IngestText->MindMap 
 
 
 
<!-- Summary -->
 
 Summary 
 
 Summary 
 
<!-- IngestText->Summary -->
 
 IngestText->Summary 
 
 
 
<!-- ThinkingHats -->
 
 ThinkingHats 
 
 ThinkingHats 
 
<!-- IngestText->ThinkingHats -->
 
 IngestText->ThinkingHats 
 
 
 
<!-- Title -->
 
 Title 
 
 Title 
 
<!-- IngestText->Title -->
 
 IngestText->Title 
 
 
 
<!-- TopicsTable -->
 
 TopicsTable 
 
 TopicsTable 
 
<!-- IngestText->TopicsTable -->
 
 IngestText->TopicsTable 
 
 
 
<!-- Report -->
 
 Report 
 
 Report 
 
<!-- MindMap->Report -->
 
 MindMap->Report 
 
 
 
<!-- Report->ExportAndOpen -->
 
 Report->ExportAndOpen 
 
 
 
<!-- Summary->Report -->
 
 Summary->Report 
 
 
 
<!-- ThinkingHats->Report -->
 
 ThinkingHats->Report 
 
 
 
<!-- Title->Report -->
 
 Title->Report 
 
 
 
<!-- TopicsTable->Report -->
 
 TopicsTable->Report 
 
 
 
<!-- TypeOfInput->IngestText -->
 
 TypeOfInput->IngestText 
 
 
 
<!-- with-title -->
 
 with-title 
 
 with-title 
 
<!-- with-title->Title -->
 
 with-title->Title

Final result:

In [8]:
#% markdown
$gCombinedSummary.nodes<Report><result>.subst(/'```html' | '```' $/):g

# «Graph» neat examples, set 3

### *LLM summary report*

## Summary

The presentation "Graph Neat Examples in Raku" by Anton Antonov showcases various graph visualizations using the Raku programming language, including undirected triangular grid graphs, bipartite grid graphs, and directed graphs with connected components. It demonstrates the use of Raku's graph package alongside JavaScript D3 and the DOT language for rendering and visualizing graphs in different formats such as SVG and interactive web-based displays. The talk concludes with a collage of star graphs and emphasizes the power and simplicity of using the DOT language for graph styling and plotting.

## Topics

<table border="1"><thead><tr><th>theme</th><th>content</th></tr></thead><tbody><tr><td align=left>Introduction and first example</td><td align=left>Presentation intro, triangular grid undirected graph, vertex degree coloring, adjacency and incidence matrices explained.</td></tr><tr><td align=left>What are NE examples</td><td align=left>NE examples defined as concise Raku code showcasing visualization, module functionalities, and computational possibilities.</td></tr><tr><td align=left>Tools and visualization methods</td><td align=left>Using Raku graph package, JavaScript D3, and GraphViz dot language for graph visualization and SVG generation.</td></tr><tr><td align=left>Second example: Bipartite grid graphs</td><td align=left>Grid graphs are bipartite; explanation of bipartite coloring, vertex sets, and code walkthrough.</td></tr><tr><td align=left>Directed graph example and connected components</td><td align=left>Randomly directed grid graph, connected components coloring, reachability explanation in directed graphs.</td></tr><tr><td align=left>Visualization in Jupyter notebooks</td><td align=left>JavaScript and SVG visualizations rendered in Jupyter notebooks using web-based technology.</td></tr><tr><td align=left>Dot language usage and vertex/node shapes</td><td align=left>Dot language for graph styling, vertex vs node terminology, random vertex shapes like hexagonal and triangular.</td></tr><tr><td align=left>Final example: Collage of star graphs</td><td align=left>40 star graphs with varied rays visualized, code possibly generated by large language models.</td></tr><tr><td align=left>Dot language advantages and styling</td><td align=left>Dot language DSL is powerful, easy to learn, allows styling preamble for consistent graph appearance.</td></tr><tr><td align=left>Conclusion</td><td align=left>Presentation wrap-up and thanks.</td></tr></tbody></table>

## Mind map

```mermaid
graph TD
    A[Graph Neat Presentation]
    A --> B[Introduction]
    B --> B1[Presenter: Anton Antonov]
    B --> B2[Date: November 13th, 20124]

    A --> C[Example 1: Undirected Triangular Grid Graph]
    C --> C1[Vertices colored by degree]
    C --> C2[Adjacency Matrix: symmetric, vertex to vertex]
    C --> C3[Incidence Matrix: vertex to edge, each column has two non-zero elements]

    A --> D[NE Examples Explained]
    D --> D1[Concise code producing compelling outputs]
    D --> D2[Showcase Raku programming and modules]
    D --> D3[Show computational possibilities]
    D --> D4[Uses Raku graph package]
    D --> D5[Visualization via JavaScript D3]
    D --> D6[Graph class dot method uses DOT language]
    D --> D7[DOT produces SVG images for visualization]

    A --> E[Example 2: Bipartite Grid Graph]
    E --> E1[Grid graphs are bipartite]
    E --> E2[Two sets of vertices, no edges within sets, edges between sets]
    E --> E3[Coloring shows bipartite property]
    E --> E4[Code generates grid graph, verifies bipartite]
    E --> E5[Derives bipartite coloring for highlights]
    E --> E6[Uses vertex coordinates for visualization]

    A --> F[Example 3: Directed Graph from Undirected Grid]
    F --> F1[Randomly directed edges]
    F --> F2[Connected components colored differently]
    F --> F3[Reachability explained]
    F --> F4[Visualized with DOT or JavaScript]
    F --> F5[JavaScript code rendered in Jupyter notebook]

    A --> G[DOT Language Visualization]
    G --> G1[Random vertex/node shapes: hexagonal, triangular]
    G --> G2[Uses both vertex shape and node shape]
    G --> G3[DOT language is powerful and easy to learn]
    G --> G4[Preamble specifies style for plots]

    A --> H[Example 4: Collage of Star Graphs]
    H --> H1[40 star graphs with 8 to 16 rays each]
    H --> H2[Subset colored for visualization]
    H --> H3[Code possibly generated by large language models]
    H --> H4[DOT visualization more regular than JavaScript]
    H --> H5[DOT language used for styles and plots]

    A --> I[Conclusion]
    I --> I1[Thank you]
```

## Thinking hats

<table>
  <tr>
    <th>Hat</th>
    <th>Perspective</th>
    <th>Feedback</th>
    <th>New Ideas</th>
  </tr>
  <tr>
    <td>Yellow Hat</td>
    <td>Benefits and creativity</td>
    <td>This presentation showcases an excellent use of Raku and graph theory concepts to create visually compelling and informative examples. The integration with JavaScript D3 and DOT language leverages powerful visualization tools, making abstract graph properties intuitive and accessible. The use of vertex coloring to represent degrees or connected components is a creative way to instantly communicate complex graph characteristics. This approach can inspire learners and developers to explore graph algorithms interactively and visually.</td>
    <td>Expand the presentation to include interactive elements where users can manipulate graphs in real-time, add animations to demonstrate graph transformations, create tutorial videos explaining each graph concept visually, or develop a web-based playground for experimenting with Raku graph code and visualizations.</td>
  </tr>
  <tr>
    <td>Gray Hat</td>
    <td>Cynicism and skepticism</td>
    <td>While the presentation sounds impressive, there is a hint of overcomplication and jargon that might alienate newcomers. The heavy reliance on multiple technologies (Raku, JavaScript D3, DOT language) could be seen as a way to mask the simplicity of the underlying concepts with flashy visuals. Also, the claim that some code was generated by large language models without thorough verification might introduce errors or inconsistencies. There's a subtle bias toward presenting Raku as uniquely capable, which might be more about promoting the language than about practical superiority.</td>
    <td>Include a critical comparison with other graph visualization tools to validate Raku's advantages, add disclaimers about the limitations or potential errors from AI-generated code, simplify explanations to avoid jargon overload, or provide a critique section acknowledging where the approach might fall short or be less efficient.</td>
  </tr>
</table>

---

## Partial evaluation

Drop the results in `LLM::Graph` computed above:

In [9]:
$gCombinedSummary.clear

LLM::Graph(size => 9, nodes => ExportAndOpen, IngestText, MindMap, Report, Summary, ThinkingHats, Title, TopicsTable, TypeOfInput)

Here the are normalized nodes without results:

In [10]:
.say for |$gCombinedSummary.nodes

TopicsTable => {eval-function => sub { }, spec-type => (Routine), wrapper => Routine::WrapHandle.new}
Report => {eval-function => sub { }, spec-type => (Callable)}
IngestText => {eval-function => sub { }, spec-type => (Callable)}
MindMap => {eval-function => sub { }, spec-type => (Routine), wrapper => Routine::WrapHandle.new}
ExportAndOpen => {eval-function => sub { }, spec-type => (Callable), test-function => -> $export-and-open = Bool::True { #`(Block|4593373995464) ... }}
ThinkingHats => {eval-function => sub { }, spec-type => (Routine), wrapper => Routine::WrapHandle.new}
Summary => {eval-function => sub { }, spec-type => (Routine), wrapper => Routine::WrapHandle.new}
TypeOfInput => {eval-function => sub { }, spec-type => (Routine), wrapper => Routine::WrapHandle.new}
Title => {eval-function => sub { }, spec-type => (Callable)}


Here all results are pre-assigned as arguments:

In [11]:
$gCombinedSummary.eval({
    '$_' => $url, 
    Title => '«Graph» neat examples, set 3', 
    :export-and-open,
    TypeOfInput => 'Other',
    Summary => 'In brief',
    IngestText => 'Ingest text',
    TopicsTable => '["TopicsTable"]',
    ThinkingHats => 'Thinking hats',
    MindMap => 'Mind-map graph'
})

LLM::Graph(size => 9, nodes => ExportAndOpen, IngestText, MindMap, Report, Summary, ThinkingHats, Title, TopicsTable, TypeOfInput)

-----

## References

### Blog posts

[AA1] Anton Antonov,
["Parameterized Literate Programming"](https://rakuforprediction.wordpress.com/2025/06/21/parameterized-literate-programming/),
(2025),
[RakuForPrediction at WordPress](https://rakuforprediction.wordpress.com).

### Notebooks

[AAn1] Anton Antonov,
["LLM comprehensive summary template for large texts"](https://community.wolfram.com/groups/-/m/t/3448842),
(2025),
[Wolfram Community](https://community.wolfram.com).

### Packages

[AAp1] Anton Antonov, 
[LLM::Graph, Raku package](https://github.com/antononcube/Raku-LLM-Graph),
(2025),
[GitHub/antononcube](https://github.com/antononcube).